# Hierarchical Context Designs for Polymarket

This notebook is a research demo for idea 3: instead of treating each market as an isolated time series, we model a target market `A` together with non-local context.

The goal here is not to build the final architecture. The goal is to compare several hierarchy choices and measure whether compressed context can preserve useful information for terminal forecasting.


## Notation

- `A`: the target market whose final outcome we want to predict.
- `B`: related market context, usually sibling markets from the same family or semantically close markets.
- `E`: external context, here represented by external covariates such as `BTC/USD` and `ETH/USD`.
- `raw(A)`: raw local features for the target market, including its current probability, recent dynamics, and age/progress features.
- `raw(B)`: raw context features aggregated from related markets.
- `raw(E)`: raw external time-series features joined as of the prediction cutoff.
- `encoded(B)`: a compact representation of related-market context, implemented here with PCA as a simple proxy encoder.
- `encoded(B,E)`: a compact representation of all non-local context.

Main ladder:

1. predict `outcome(A)` from `raw(A)`
2. predict `outcome(A)` from `raw(A) + raw(B)`
3. predict `outcome(A)` from `raw(A) + encoded(B)`
4. predict `outcome(A)` from `raw(A) + raw(E)`
5. predict `outcome(A)` from `raw(A) + raw(B) + raw(E)`
6. predict `outcome(A)` from `raw(A) + encoded(B,E)`


## Environment and Imports

This cell imports the notebook dependencies and wires the repository paths into `sys.path`.

The logic is deliberate: we reuse the benchmark helpers that already define the terminal dataset, so the hierarchy comparison stays aligned with the benchmark story rather than inventing a separate preprocessing pipeline.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from polymarket_research import PolymarketDataset
from polymarket_research.benchmarks.covariate_utils import (
    load_external_covariates,
    pivot_covariates_to_wide,
)
from polymarket_research.benchmarks.dataset_utils import rolling_time_splits
from polymarket_research.utils import setup_root

REPO_ROOT = setup_root()


## Configuration and Helper Functions

This cell defines the working domains, the horizon set, and a few helper functions for weak family construction, context aggregation, and simple PCA-based encoding.

The key modeling decision here is to keep the notebook lightweight. We do not train a learned encoder yet; instead, we use PCA as a proxy so we can isolate the question: can non-local context be compressed at all without losing too much predictive value?


In [ ]:
DB_PATH = DEFAULT_DB_PATH
DOMAINS = ('crypto', 'politics', 'geopolitics', 'technology', 'finance_economy')
MAX_MARKETS_PER_DOMAIN = 250
MIN_PROBABILITY_ROWS = 288
HORIZONS = (24, 72, 168)
CONTEXT_PCA_DIM = 6
EXTERNAL_PATH = REPO_ROOT / 'cached_data' / 'external_covariates'


def parse_listish(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except Exception:
        pass
    if '|' in text:
        parts = text.split('|')
    elif ',' in text:
        parts = text.split(',')
    else:
        parts = [text]
    return [part.strip() for part in parts if part.strip()]


def normalize_text(value: str) -> str:
    text = str(value or '').lower()
    text = re.sub(r'[^a-z0-9\s]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def build_family_id(question: str, domain: str, tags) -> str:
    norm_q = normalize_text(question)
    norm_tags = [normalize_text(tag) for tag in parse_listish(tags)]
    tokens = [tok for tok in norm_q.split() if tok not in {'will', 'the', 'a', 'an', 'be', 'is', 'are', 'to', 'of', 'by', 'in'}]
    key = ' '.join(tokens[:6]) if tokens else norm_q[:48]
    tag_key = '|'.join(sorted(norm_tags[:3]))
    return f"{domain}::{tag_key}::{key}".strip(':')


def latest_context_snapshot(probabilities_df: pd.DataFrame, market_id: str, cutoff: pd.Timestamp):
    panel = probabilities_df.loc[(probabilities_df['market_id'] == market_id) & (probabilities_df['timestamp_utc'] <= cutoff)]
    if panel.empty:
        return None
    return panel.iloc[-1]


def build_family_context_features(dataset: pd.DataFrame, probabilities_df: pd.DataFrame, market_meta: pd.DataFrame) -> pd.DataFrame:
    family_map = market_meta.groupby('family_id')['market_id'].apply(list).to_dict()
    rows = []
    for row in dataset[['market_id', 'cutoff_timestamp_utc']].itertuples(index=False):
        family_id = market_meta.loc[market_meta['market_id'] == row.market_id, 'family_id'].iloc[0]
        related_ids = [mid for mid in family_map.get(family_id, []) if mid != row.market_id]
        related_probs = []
        related_trade_flags = []
        for related_id in related_ids:
            snap = latest_context_snapshot(probabilities_df, related_id, row.cutoff_timestamp_utc)
            if snap is None:
                continue
            related_probs.append(float(snap['yes_probability']))
            related_trade_flags.append(float(snap['observed_trade']))
        if related_probs:
            prob_arr = np.asarray(related_probs, dtype=float)
            trade_arr = np.asarray(related_trade_flags, dtype=float)
            out = {
                'market_id': row.market_id,
                'cutoff_timestamp_utc': row.cutoff_timestamp_utc,
                'family_related_count': float(len(prob_arr)),
                'family_prob_mean': float(np.mean(prob_arr)),
                'family_prob_std': float(np.std(prob_arr)),
                'family_prob_min': float(np.min(prob_arr)),
                'family_prob_max': float(np.max(prob_arr)),
                'family_prob_gap': float(np.max(prob_arr) - np.min(prob_arr)),
                'family_trade_share_mean': float(np.mean(trade_arr)),
            }
        else:
            out = {
                'market_id': row.market_id,
                'cutoff_timestamp_utc': row.cutoff_timestamp_utc,
                'family_related_count': 0.0,
                'family_prob_mean': np.nan,
                'family_prob_std': np.nan,
                'family_prob_min': np.nan,
                'family_prob_max': np.nan,
                'family_prob_gap': np.nan,
                'family_trade_share_mean': np.nan,
            }
        rows.append(out)
    return pd.DataFrame(rows)


def build_external_feature_frame(path: Path) -> pd.DataFrame:
    covariates = load_external_covariates(path)
    wide = pivot_covariates_to_wide(covariates, value_col='value')
    feature_frame = add_lagged_covariate_features(wide, lags=(1, 6, 12), pct_change=True)
    return feature_frame


def safe_auc(y_true, p_pred):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, p_pred)


def clipped(p):
    return np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)


def fit_pca_on_train(train_df: pd.DataFrame, test_df: pd.DataFrame, cols, n_components: int, prefix: str):
    cols = [col for col in cols if col in train_df.columns]
    if not cols:
        return pd.DataFrame(index=train_df.index), pd.DataFrame(index=test_df.index)
    n_components = max(1, min(int(n_components), len(cols), max(1, len(train_df) - 1)))
    imputer = SimpleImputer(strategy='median')
    scaler = StandardScaler()
    train_x = scaler.fit_transform(imputer.fit_transform(train_df[cols]))
    test_x = scaler.transform(imputer.transform(test_df[cols]))
    pca = PCA(n_components=n_components, random_state=0)
    train_z = pca.fit_transform(train_x)
    test_z = pca.transform(test_x)
    train_out = pd.DataFrame(train_z, index=train_df.index, columns=[f'{prefix}_{i+1}' for i in range(train_z.shape[1])])
    test_out = pd.DataFrame(test_z, index=test_df.index, columns=[f'{prefix}_{i+1}' for i in range(test_z.shape[1])])
    return train_out, test_out


## Load Markets and Build the Terminal Dataset

Here we load markets across the available domains, fetch their probability histories, build the multi-horizon terminal dataset, and then attach two kinds of non-local context:

- family-style related-market summaries (`B`)
- external covariates (`E`)

This is the main data-construction step for the hierarchy experiment. The downstream comparison only makes sense if all variants are evaluated on the same snapshots and targets.


In [ ]:
DATASET_ARTEFACT_DIR = REPO_ROOT / 'research_notebooks' / 'running_artefacts'

dataset = PolymarketDataset.from_parquet(DATASET_ARTEFACT_DIR)
markets = dataset.markets.copy()
probabilities = dataset.probabilities.copy()


## Hierarchy variants

This notebook keeps `A` in raw form and only compresses non-local context. That is the most practical first version of the hierarchy idea.

- `raw(A)`: local market only.
- `raw(A) + raw(B)`: local market plus raw related-market context.
- `raw(A) + encoded(B)`: local market plus compressed related-market context.
- `raw(A) + raw(E)`: local market plus raw external context.
- `raw(A) + raw(B) + raw(E)`: full raw context.
- `raw(A) + encoded(B,E)`: local market plus compressed non-local context.

If `encoded(B,E)` is competitive with `raw(B) + raw(E)`, that is evidence that the non-local market state can be summarized compactly without losing too much task-relevant information.


## Evaluate Hierarchy Variants

This cell defines the feature blocks and runs the main experiment ladder:

- `raw(A)`
- `raw(A) + raw(B)`
- `raw(A) + encoded(B)`
- `raw(A) + raw(E)`
- `raw(A) + raw(B) + raw(E)`
- `raw(A) + encoded(B,E)`

We keep `A` in raw form and only compress non-local context. That is the cleanest first test of the representation idea: whether context can be summarized compactly while preserving value for terminal forecasting.


In [ ]:
local_a_cols = [
    'market_price_baseline',
    'current_yes_probability',
    'confidence_margin',
    'snapshot_staleness_hours',
    'observed_trade_now',
    'trade_count_now',
    'total_size_now',
    'last_trade_price_now',
    'lookback_24h_rows',
    'lookback_24h_observed_trade_share',
    'lookback_24h_trade_count_sum',
    'lookback_24h_total_size_sum',
    'lookback_24h_yes_probability_change',
    'lookback_24h_volatility',
    'lookback_24h_abs_move_mean',
    'lookback_24h_abs_move_max',
    'lookback_168h_rows',
    'lookback_168h_observed_trade_share',
    'lookback_168h_trade_count_sum',
    'lookback_168h_total_size_sum',
    'lookback_168h_yes_probability_change',
    'lookback_168h_volatility',
    'lookback_168h_abs_move_mean',
    'lookback_168h_abs_move_max',
    'hours_to_resolution',
    'market_age_hours',
    'life_progress',
    'horizon_hours',
]

raw_b_cols = [
    'family_related_count',
    'family_prob_mean',
    'family_prob_std',
    'family_prob_min',
    'family_prob_max',
    'family_prob_gap',
    'family_trade_share_mean',
]

raw_e_cols = [
    col for col in terminal.columns
    if col.startswith('btc_usd_') or col.startswith('eth_usd_')
]

domain_dummies = pd.get_dummies(terminal['primary_domain'], prefix='domain', dtype=float)
work = pd.concat([terminal.copy(), domain_dummies], axis=1)
local_a_cols = local_a_cols + list(domain_dummies.columns)

variants = {
    'raw(A)': {'base': local_a_cols, 'encode': []},
    'raw(A)+raw(B)': {'base': local_a_cols + raw_b_cols, 'encode': []},
    'raw(A)+encoded(B)': {'base': local_a_cols, 'encode': raw_b_cols},
    'raw(A)+raw(E)': {'base': local_a_cols + raw_e_cols, 'encode': []},
    'raw(A)+raw(B)+raw(E)': {'base': local_a_cols + raw_b_cols + raw_e_cols, 'encode': []},
    'raw(A)+encoded(B,E)': {'base': local_a_cols, 'encode': raw_b_cols + raw_e_cols},
}

metric_rows = []
for train_df, test_df, meta in rolling_time_splits(work, time_col='end_date', n_splits=4, min_train_fraction=0.5):
    y_train = train_df['target'].astype(int).to_numpy()
    y_test = test_df['target'].astype(int).to_numpy()
    for variant_name, spec in variants.items():
        base_cols = [col for col in spec['base'] if col in train_df.columns]
        train_parts = [train_df[base_cols].copy()] if base_cols else []
        test_parts = [test_df[base_cols].copy()] if base_cols else []
        if spec['encode']:
            train_z, test_z = fit_pca_on_train(train_df, test_df, spec['encode'], CONTEXT_PCA_DIM, prefix='z')
            train_parts.append(train_z)
            test_parts.append(test_z)
        x_train = pd.concat(train_parts, axis=1)
        x_test = pd.concat(test_parts, axis=1)
        model = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(max_iter=2000, C=1.0)),
        ])
        model.fit(x_train, y_train)
        p_test = model.predict_proba(x_test)[:, 1]
        fold_metrics = {
            'variant': variant_name,
            'fold': meta['fold'],
            'train_size': meta['train_size'],
            'test_size': meta['test_size'],
            'log_loss': log_loss(y_test, clipped(p_test)),
            'brier': brier_score_loss(y_test, clipped(p_test)),
            'roc_auc': safe_auc(y_test, p_test),
        }
        metric_rows.append(fold_metrics)
        fold_frame = test_df[['primary_domain', 'horizon_name']].copy()
        fold_frame['target'] = y_test
        fold_frame['pred'] = p_test
        fold_frame['variant'] = variant_name
        for (domain, horizon), group in fold_frame.groupby(['primary_domain', 'horizon_name'], dropna=False):
            metric_rows.append({
                'variant': variant_name,
                'fold': meta['fold'],
                'train_size': meta['train_size'],
                'test_size': len(group),
                'primary_domain': domain,
                'horizon_name': horizon,
                'log_loss': log_loss(group['target'], clipped(group['pred'])),
                'brier': brier_score_loss(group['target'], clipped(group['pred'])),
                'roc_auc': safe_auc(group['target'], group['pred']),
            })

metrics = pd.DataFrame(metric_rows)
overall = metrics.loc[metrics['primary_domain'].isna()].copy() if 'primary_domain' in metrics.columns else metrics.copy()
overall_summary = (
    overall.groupby('variant', dropna=False)[['log_loss', 'brier', 'roc_auc']]
    .mean()
    .sort_values('log_loss')
    .reset_index()
)
display(overall_summary)


## Overall Comparison Plot

This plot summarizes average performance over rolling splits.

The interpretation is simple:

- if `raw(A) + raw(B)` beats `raw(A)`, related markets matter;
- if `raw(A) + encoded(B)` stays close to `raw(A) + raw(B)`, context compression is plausible;
- if `raw(A) + encoded(B,E)` stays close to the full raw-context model, a compact non-local latent state becomes a realistic design target.


In [ ]:
plot_summary = overall_summary.copy()
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=plot_summary, x='log_loss', y='variant', palette='crest', ax=axes[0])
axes[0].set_title('Mean Log Loss by Hierarchy Variant')
axes[0].set_xlabel('Lower is better')
axes[0].set_ylabel('')

sns.barplot(data=plot_summary.sort_values('roc_auc', ascending=False), x='roc_auc', y='variant', palette='flare', ax=axes[1])
axes[1].set_title('Mean ROC-AUC by Hierarchy Variant')
axes[1].set_xlabel('Higher is better')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()


## Slice-Level Comparison

Average metrics can hide where hierarchy actually helps. This cell breaks the results down by horizon and domain.

That matters conceptually because the representation story is strongest if context helps in hard regimes, across multiple domains, or at earlier horizons where the target market is less degenerate.


In [ ]:
by_slice = metrics.loc[metrics['primary_domain'].notna()].copy()
slice_summary = (
    by_slice.groupby(['variant', 'primary_domain', 'horizon_name'], dropna=False)[['log_loss', 'roc_auc']]
    .mean()
    .reset_index()
)
display(slice_summary.sort_values(['primary_domain', 'horizon_name', 'log_loss']).head(18))

fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=False)
sns.lineplot(
    data=slice_summary,
    x='horizon_name',
    y='log_loss',
    hue='variant',
    style='primary_domain',
    markers=True,
    dashes=False,
    ax=axes[0],
)
axes[0].set_title('Log Loss by Horizon and Domain')
axes[0].set_xlabel('Horizon')
axes[0].set_ylabel('Lower is better')

sns.lineplot(
    data=slice_summary,
    x='horizon_name',
    y='roc_auc',
    hue='variant',
    style='primary_domain',
    markers=True,
    dashes=False,
    ax=axes[1],
)
axes[1].set_title('ROC-AUC by Horizon and Domain')
axes[1].set_xlabel('Horizon')
axes[1].set_ylabel('Higher is better')
plt.tight_layout()
plt.show()


## How to read the results

- If `raw(A) + raw(B)` beats `raw(A)`, then related-market context matters.
- If `raw(A) + encoded(B)` is close to `raw(A) + raw(B)`, then related-market context can be compressed without losing much task-relevant information.
- If `raw(A) + raw(E)` helps, then external signals contain incremental information beyond the target market itself.
- If `raw(A) + encoded(B,E)` is close to `raw(A) + raw(B) + raw(E)`, then a compact non-local latent state is plausible.

For a stronger next step, replace the PCA proxy with a learned context encoder and repeat the same evaluation ladder.
